> **SUPERSEDED for interactive use by `notebooks/V1_Colab.ipynb`** — Colab gives every notebook a fresh VM, so the environment (deps, Drive mount, repo clone, env vars) does NOT persist between notebooks. Run everything in one: `notebooks/V1_Colab.ipynb`.
# 00 — Setup

Environment check, Drive mount (skippable), repo bootstrap, dependency install, weight download (opt-in), and a model smoke.

**All parameters come from forms/env — no manual editing.** Set `COLAB_DRIVE_SKIP=1` to run without Drive (local tmp dirs, used by CI/`nbconvert`).

In [ ]:
import os, subprocess, sys, tempfile
import shutil

DRIVE_SKIP = os.environ.get("COLAB_DRIVE_SKIP", "0") == "1"
REPO_DIR = os.path.abspath(os.getcwd())
print("repo dir:", REPO_DIR)
print("drive skip:", DRIVE_SKIP)

In [ ]:
#@title GPU detection
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    print("gpu:", torch.cuda.get_device_name(0), "| compute capability:", cc)
    if cc[0] >= 8:
        print("PROFILE: FULL (sm_80+) -> bf16 precision + fused mamba kernels")
    else:
        print("PROFILE: DEV (T4, sm_75) -> fp16 precision + eager mamba fallback")
else:
    print("PROFILE: CPU/dev -> fp32, eager mamba (no GPU)")
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
#@title Drive mount + env (skipped with COLAB_DRIVE_SKIP=1)
import pathlib

if not DRIVE_SKIP:
    from google.colab import drive
    drive.mount("/content/drive")
    COLAB_DRIVE = "/content/drive/MyDrive/ssm-phylo"
    for d in [COLAB_DRIVE, f"{COLAB_DRIVE}/data", f"{COLAB_DRIVE}/checkpoints",
              f"{COLAB_DRIVE}/results", f"{COLAB_DRIVE}/weights"]:
        pathlib.Path(d).mkdir(parents=True, exist_ok=True)
else:
    COLAB_DRIVE = tempfile.mkdtemp(prefix="ssm_drive_")

os.environ.update(
    COLAB_DRIVE=COLAB_DRIVE,
    DATA_DIR=f"{COLAB_DRIVE}/data",
    CKPT_DIR=f"{COLAB_DRIVE}/checkpoints",
    RESULTS_DIR=f"{COLAB_DRIVE}/results",
    PROT_MAMBA_CKPT=f"{COLAB_DRIVE}/weights/protmamba",
    LOCAL_CKPT_DIR="/content/ckpts" if not DRIVE_SKIP else f"{COLAB_DRIVE}/local_ckpts",
    LOCAL_DATA_DIR="/content/data" if not DRIVE_SKIP else f"{COLAB_DRIVE}/local_data",
)
for d in [os.environ["LOCAL_CKPT_DIR"], os.environ["LOCAL_DATA_DIR"]]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("COLAB_DRIVE:", COLAB_DRIVE)
print("DATA_DIR:", os.environ["DATA_DIR"])

In [ ]:
#@title Repo bootstrap (clone into /content when not already there)
REPO_URL = "https://github.com/amichae2/SSM-Phylo.git"  #@param {type:"string"}
if not DRIVE_SKIP and not os.path.isdir("/content/ssm-phylo"):
    subprocess.run(["git", "clone", REPO_URL, "/content/ssm-phylo"], check=True)
    os.chdir("/content/ssm-phylo")
    REPO_DIR = "/content/ssm-phylo"
print("using repo at:", REPO_DIR)

In [ ]:
#@title colab_setup.sh (idempotent; never fails on mamba-ssm build failure)
setup = subprocess.run(
    ["bash", f"{REPO_DIR}/scripts/colab_setup.sh"],
    env={**os.environ, "COLAB_DRIVE": COLAB_DRIVE},
    capture_output=True, text=True,
)
print(setup.stdout[-3000:])
print("setup exit:", setup.returncode)

In [ ]:
#@title Download ProtMamba weights? (degraded_protmamba mode only)
download_weights = False  #@param {type:"boolean"}
if download_weights:
    dl = subprocess.run(["bash", f"{REPO_DIR}/scripts/download_weights.sh"],
                        env=os.environ, capture_output=True, text=True)
    print(dl.stdout[-2000:])
else:
    print("skipped — from_scratch needs no weights (license-clean default)")

In [ ]:
#@title Verify imports + tiny model smoke
sys.path.insert(0, REPO_DIR)
import ssm_phylo
from ssm_phylo.models.encoder import build_encoder
from ssm_phylo.models.head import PhyloModel
from types import SimpleNamespace

cfg = SimpleNamespace(
    d_model=32, n_layer=2, vocab_size=38,
    encoder=SimpleNamespace(kind="from_scratch", checkpoint_dir=None,
        mamba={"state_size": 4, "time_step_rank": 8, "conv_kernel": 3, "expand": 2},
        ptm_model_id="ChatterjeeLab/PTM-Mamba"),
)
model = PhyloModel(build_encoder(cfg), d_emb=16, max_dist=3.0)
import torch
tok = torch.randint(0, 38, (1, 64))
spans = torch.tensor([[[0, 32], [33, 64]]])
dm, embs = model(tok, spans, torch.ones(1, 2, dtype=torch.bool))
print("dm:", tuple(dm.shape), "embs:", tuple(embs.shape), "ssm_phylo", ssm_phylo.__version__)
print("SETUP OK")